# Step 9: Experiment Tracking with MLflow

**SageMaker Unified Studio Component**: MLflow

**What you'll learn**: Track experiments for reproducibility

In [ ]:
# Ensure sagemaker-mlflow plugin is installed for ARN-based tracking
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "sagemaker-mlflow", "--quiet", "--upgrade"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"pip install failed: {result.stderr}")
else:
    print("sagemaker-mlflow installed/upgraded")

# Check installed version
try:
    import importlib.metadata
    ver = importlib.metadata.version("sagemaker-mlflow")
    print(f"sagemaker-mlflow version: {ver}")
except Exception:
    print("WARNING: sagemaker-mlflow not found after install")

In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

bucket_name = os.getenv('BUCKET_NAME', 'sagemaker-unified-overheat-demo-658203403846')

## Setup MLflow

In [ ]:
# Parameters (injected by workflow via papermill)
mlflow_tracking_uri = "arn:aws:sagemaker:eu-west-1:658203403846:mlflow-app/app-IN74ELWDTMBI"

In [ ]:
# Configure MLflow tracking - try ARN first, fall back to URL resolution via boto3
import traceback

try:
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment("machine-overheat")
    print(f"MLflow tracking URI: {mlflow_tracking_uri}")
    print("MLflow experiment: machine-overheat")
except Exception as e:
    print(f"ARN-based tracking failed: {e}")
    print("Falling back to boto3 URL resolution...")
    
    # Extract app ID from ARN and construct the HTTPS URL
    # ARN format: arn:aws:sagemaker:REGION:ACCOUNT:mlflow-app/APP_ID
    import re, boto3
    match = re.search(r'arn:aws:sagemaker:([^:]+):[^:]+:mlflow-app/(.+)', mlflow_tracking_uri)
    if match:
        region, app_id = match.group(1), match.group(2)
        tracking_url = f"https://{app_id}.mlflow.sagemaker.{region}.app.aws"
        print(f"Resolved tracking URL: {tracking_url}")
        
        # Use SigV4 auth via environment
        os.environ['MLFLOW_TRACKING_URI'] = tracking_url
        os.environ['MLFLOW_TRACKING_AWS_SIGV4'] = 'true'
        os.environ['AWS_DEFAULT_REGION'] = region
        
        mlflow.set_tracking_uri(tracking_url)
        mlflow.set_experiment("machine-overheat")
        print(f"MLflow experiment set with URL: {tracking_url}")
    else:
        raise RuntimeError(f"Cannot parse MLflow ARN: {mlflow_tracking_uri}") from e

## Load Data

In [ ]:
s3_path = f's3://{bucket_name}/data/features/features.parquet'
df = pd.read_parquet(s3_path)

X = df[['temperature', 'temp_diff']]
y = df['overheat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Train and Log with MLflow

In [ ]:
with mlflow.start_run(run_name="logistic_regression_v1"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(f"✓ Run logged with accuracy: {accuracy:.3f}")